[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/en_clase/S12_05_transformers_atencion_en_clase.ipynb)

# S12 · 05 · Transformers y LLM: tokenización y atención paso a paso · versión en clase

**Tiempo estimado:** 40 min en clase · **Sprint 12** (UD 4.9) · Cliente: **TurisData Canarias**

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/codigo_explicado/S12_05_transformers_atencion.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Lo que vas a aprender**
1. Qué es un **token** y cómo trocea un texto un modelo de lenguaje (idea de BPE).
2. Cómo funciona la **autoatención** (*self-attention*), calculada a mano con `numpy` y números pequeños.
3. Qué hace la **máscara causal** y por qué un LLM genera palabra a palabra; y qué controla la **temperatura**.
4. (Opcional, con internet) Usar modelos preentrenados de Hugging Face: sentimiento y clasificación *zero-shot*.

**Contexto.** Antes de que TurisData use un LLM como asistente, su equipo debe entender qué hay dentro de la "caja negra":
no es magia, es **matrices y probabilidades**.

> 🏫 **Versión «en clase»** · ⏱ ≈ 40 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **3, 4, 5**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_05_transformers_atencion.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import re, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
np.set_printoptions(precision=2, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False, "axes.spines.right": False})

## 1. Tokenización: cómo "lee" un LLM
Un LLM no lee letras ni palabras completas, sino **tokens**: trozos frecuentes de texto.
- Trocear por **palabras** genera vocabularios enormes y no sabe qué hacer con palabras nuevas.
- Trocear por **letras** da secuencias larguísimas.
- **BPE** (*byte pair encoding*) es un punto medio: parte de letras y va **fusionando los pares más frecuentes**
  hasta formar trozos útiles ("desayun" + "o").

Vamos a implementar un BPE de juguete con un mini corpus.

In [ ]:
corpus = ["desayuno", "desayunos", "desayunar", "limpio", "limpieza", "limpiar", "limpiamos", "desayunamos"]

def contar_pares(palabras):
    """palabras: lista de listas de símbolos. Devuelve un Counter con la frecuencia de cada par de símbolos consecutivos."""
    pares = Counter()
    for simbolos in palabras:
        for par in zip(simbolos, simbolos[1:]):
            pares[par] += 1
    return pares

Observa cómo aparecen trozos como `desayun` o `limpi`: el modelo ha "descubierto" raíces sin saber español.
Una palabra nueva como "desayunando" se trocearía reutilizando esos trozos.


**Por qué importa en la práctica:** los LLM cobran y se limitan **por tokens**, y el español suele gastar más tokens que el inglés.

## 2. Autoatención paso a paso (con números pequeños)
**Idea.** Al leer "El hotel es limpio pero ruidoso", para entender *ruidoso* conviene mirar a *hotel*.
La **atención** deja que cada palabra decida **cuánto mirar a las demás** y mezcle su información.
Se hace con tres versiones de cada palabra:
- **Q (query, "pregunta")**: lo que esta palabra busca.
- **K (key, "etiqueta")**: lo que cada palabra ofrece.
- **V (value, "contenido")**: la información que se pasa.

Pasos: 1) Q·Kᵀ = *puntuaciones* de compatibilidad; 2) dividir por √d; 3) **softmax** por filas → pesos que suman 1;
4) salida = pesos · V.

Usamos **3 palabras** y vectores de tamaño 4, con números enteros para poder seguir las cuentas a mano.

In [ ]:
palabras = ["hotel", "muy", "limpio"]
# Embeddings inventados (una fila por palabra, 4 dimensiones)
X = np.array([[1., 0., 1., 0.],
              [0., 2., 0., 2.],
              [1., 1., 0., 1.]])

# Matrices de proyección (4 → 2). En un modelo real se APRENDEN durante el entrenamiento.
Wq = np.array([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
Wk = np.array([[1., 0.], [0., 1.], [0., 1.], [1., 0.]])
Wv = np.array([[1., 2.], [0., 1.], [1., 0.], [0., 1.]])

Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("Q =\n", Q, "\nK =\n", K, "\nV =\n", V)

**Paso 1: puntuaciones** = Q · Kᵀ. La fila *i*, columna *j* dice cuánto le interesa la palabra *i* la palabra *j*.

In [ ]:
puntuaciones = Q @ K.T
print(puntuaciones)

**Paso 2: escalar** por √d (d = 2 aquí) para que los números no crezcan demasiado, y **paso 3: softmax**.
El softmax convierte cada fila en probabilidades (positivas, suman 1): `exp(x) / suma(exp(x))`.

**Ejercicio 3 · 🏫 en clase.** Escribe `softmax(m)` que aplique el softmax **por filas** a una matriz (resta el máximo de cada fila antes
de exponenciar, por estabilidad numérica).

In [ ]:
def softmax(m):
    # TODO: resta el máximo de cada fila (keepdims=True), aplica exp y divide cada fila entre su suma
    ...

_p = softmax(np.array([[1., 2., 3.], [0., 0., 0.]]))
assert np.allclose(_p.sum(axis=1), 1), "Cada fila de softmax debe sumar 1 (usa axis=1)."
assert np.allclose(_p[1], 1 / 3), "Con entradas iguales, todas las probabilidades valen lo mismo."
assert _p[0, 2] > _p[0, 1] > _p[0, 0], "Un valor mayor debe recibir más probabilidad."

In [ ]:
d = K.shape[1]
pesos = softmax(puntuaciones / np.sqrt(d))
print("Pesos de atención (cada fila suma 1):\n", pesos)

**Paso 4: salida** = pesos · V. Cada palabra se convierte en una **mezcla** de los contenidos de las demás, según sus pesos.
Ese vector ya "sabe" del contexto.

**Ejercicio 4 · 🏫 en clase.** Completa `atencion(X, Wq, Wk, Wv)` que devuelve `(salida, pesos)` siguiendo los 4 pasos.

In [ ]:
def atencion(X, Wq, Wk, Wv):
    # TODO: sigue los 4 pasos: proyecciones, puntuaciones escaladas, softmax y mezcla de V
    ...

salida, pesos2 = atencion(X, Wq, Wk, Wv)
print("Salida:\n", salida)
assert salida.shape == (3, 2), "La salida debe tener una fila por palabra y 2 columnas (dimensión de V)."
assert np.allclose(pesos2, pesos), "Los pesos deben coincidir con los calculados paso a paso arriba."
assert np.allclose(salida, pesos @ V), "salida = pesos @ V."

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(pesos, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(palabras); ax.set_yticks(range(3)); ax.set_yticklabels(palabras)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{pesos[i, j]:.2f}", ha="center", va="center", color="black" if pesos[i, j] < .6 else "white")
ax.set_xlabel("Mira a…"); ax.set_ylabel("Palabra que pregunta"); ax.set_title("Pesos de atención")
plt.tight_layout(); plt.show()

## 3. Máscara causal: mirar solo hacia atrás
Un LLM genera texto **de izquierda a derecha**: al predecir la palabra 2, no puede ver la 3 (aún no existe).
Se consigue **poniendo −∞ en las puntuaciones de las palabras futuras** antes del softmax: su peso pasa a 0.

**Ejercicio 5 · 🏫 en clase.** Completa `atencion_causal(X, Wq, Wk, Wv)`: igual que `atencion`, pero antes del softmax
pon `-1e9` en las posiciones **por encima de la diagonal** (`np.triu(..., k=1)`).

In [ ]:
def atencion_causal(X, Wq, Wk, Wv):
    # TODO: igual que atencion, pero con una máscara booleana del triángulo superior (np.triu, k=1) puesta a -1e9 antes del softmax
    ...

_, pesos_c = atencion_causal(X, Wq, Wk, Wv)
print(pesos_c)
assert np.allclose(np.triu(pesos_c, k=1), 0), "Los pesos por encima de la diagonal (futuro) deben ser 0."
assert np.allclose(pesos_c.sum(axis=1), 1), "Cada fila sigue sumando 1."
assert np.isclose(pesos_c[0, 0], 1.0), "La primera palabra solo puede mirarse a sí misma."

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_05_transformers_atencion.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
